# Workstream C: Business

Two files and three numbers. The deck is built in Canva; this notebook produces what
goes into it.

| Output | What it is |
|---|---|
| `out/swan_mailer_top500.csv` | the 500 customers to mail |
| `out/swan_churn_risk_all.csv` | churn risk for every customer still with Swan |
| Section 6 | which sign-up metric to incentivise at $2.50 |
| Section 7 | what the mailer costs and what it has to return |
| Section 8 | how to tell next quarter whether it worked |

**The data is a snapshot of last quarter**, so 1,869 of the 7,043 rows are customers who
have already gone. Both lists are built from the 5,174 who are still here. Section 3 does
that split and explains why it matters.

## 1 · Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

Path("out").mkdir(exist_ok=True)

# Swan brand palette. Navy is sampled from the logo wordmark; teal, coral and grey
# stay as the agreed data colours. Charts sit on the same paper as the slides so
# they drop into Canva without a white box around them.
navy, slate, paper, rule = "#163e67", "#55697c", "#f7f9fb", "#dce3ea"
teal, coral, grey = "#136e78", "#d1495b", "#9aa5a8"

plt.rcParams.update({
    "font.size": 11,
    "savefig.dpi": 200,                 # 2x for the deck
    "savefig.bbox": "tight",
    "figure.facecolor": paper,
    "axes.facecolor": paper,
    "savefig.facecolor": paper,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": rule,
    "axes.labelcolor": slate,
    "xtick.color": slate,
    "ytick.color": navy,
    "text.color": navy,
    "axes.grid": True,
    "grid.color": rule,
    "axes.axisbelow": True,
})

In [ ]:
NAME = "Swan Consulting 1 - Project Data.xlsx"
PATH = next(c for c in [Path("data") / NAME, Path(NAME), Path("..") / NAME] if c.exists())
print(f"reading {PATH}")

df = pd.read_excel(PATH, sheet_name="Telco_Churn")
df["Total Charges"] = pd.to_numeric(df["Total Charges"], errors="coerce").fillna(0)
df = df.drop(columns=["Count", "Country", "State", "Churn Label"])

df["is_echeck"] = (df["Payment Method"] == "Electronic check").astype(int)
df["is_month_to_month"] = (df["Contract"] == "Month-to-month").astype(int)
df["tenure_band"] = pd.cut(df["Tenure Months"], [-1, 6, 12, 24, 48, 72],
                           labels=["0-6", "7-12", "13-24", "25-48", "49-72"])

N = len(df)
BASE = df["Churn Value"].mean()
print(f"{N:,} rows, {BASE:.1%} of them churned last quarter")

## 2 · Churn scores from B

B delivers one file, `churn_scores.csv`, with all 7,043 rows and two columns:

```
CustomerID,churn_probability
3668-QPYBK,0.3459
9237-HQITU,0.6247
```

Three requirements, in order of how badly they break things:

1. **Out-of-fold.** Every row scored by a fold that did not train on it. A model scoring
   its own training data gives inflated probabilities. This is the one that silently
   ruins the deliverable.
2. **Calibrated.** A customer scored 0.7 should churn about 70% of the time. The service
   team reads deliverable 2 as a risk number, so it has to mean what it says. Check with a
   reliability curve and Brier score, not AUC, which is unchanged by calibration.
3. **All 7,043 rows, CustomerID untouched**, no index column.

In [ ]:
# Required, with no fallback on purpose. A placeholder here would quietly produce
# two client-facing CSVs built on invented numbers.
SCORES = next((c for c in [Path("churn_scores.csv"), Path("data/churn_scores.csv"),
                           Path("../churn_scores.csv")] if c.exists()), None)
if SCORES is None:
    raise FileNotFoundError(
        "churn_scores.csv not found. Run B - logistic_regression.ipynb and export "
        "oof_proba alongside df_clean['CustomerID'] before running this notebook.")

scores = pd.read_csv(SCORES)
scored = df.merge(scores, on="CustomerID", how="left")

assert len(scored) == N, "row count changed on merge"
assert scored["CustomerID"].is_unique, "duplicate CustomerIDs"
assert scored["churn_probability"].notna().all(), "some customers have no score"
assert scored["churn_probability"].between(0, 1).all(), "probabilities outside 0 to 1"
print(f"scores from {SCORES}: {len(scores):,} rows, validated")

In [ ]:
BANDS = [("Critical", 0.60), ("High", 0.40), ("Medium", 0.20), ("Low", 0.00)]

def band(p):
    for name, floor in BANDS:
        if p >= floor:
            return name

scored["risk_band"] = scored["churn_probability"].apply(band)

check = scored.groupby("risk_band")["Churn Value"].agg(n="size", observed_churn="mean")
check.reindex([b[0] for b in BANDS])

Every band's observed churn should sit inside its own range, and it does. That is what
lets the service team read "High" as meaning roughly a coin flip rather than just "worse
than the band below".

### How good is the ranking?

This is measured on **all 7,043 rows**, because that is the only place outcomes are known.
It answers: if we had run this model at the start of last quarter, how many of its top 500
would have actually left?

In [ ]:
# Rank all 7,043 as they stood at the start of last quarter, take the top k,
# and count how many of those people actually left.
order = np.argsort(-scored["churn_probability"].values)
outcomes = scored["Churn Value"].values[order]
total_churn = int(scored["Churn Value"].sum())

rows = []
for k in [100, 250, 500, 1000, 1500]:
    caught = int(outcomes[:k].sum())
    rows.append({"list size": k, "churners found": caught, "precision": caught / k,
                 "lift vs random": caught / k / BASE, "share of all churn": caught / total_churn})
backtest = pd.DataFrame(rows)
backtest.style.format({"list size": "{:,}", "churners found": "{:,}", "precision": "{:.1%}",
                       "lift vs random": "{:.1f}x", "share of all churn": "{:.1%}"})

In [ ]:
ks = np.arange(1, len(outcomes) + 1)
captured = np.cumsum(outcomes) / total_churn

fig, ax = plt.subplots(figsize=(9.5, 4.4))
ax.plot(ks, captured, color=teal, lw=2.5, label="Our model")
ax.plot([0, len(ks)], [0, 1], color=grey, ls="--", lw=1.5, label="Contacting people at random")
ax.axvline(500, color=coral, lw=1.4, ls=":")
ax.plot([500], [captured[499]], "o", color=coral, ms=8, zorder=5)
ax.annotate(f"500 mailers reach {captured[499]:.0%} of everyone who left",
            xy=(500, captured[499]), xytext=(1100, captured[499] - 0.13),
            fontsize=10.5, color=coral,
            arrowprops=dict(arrowstyle="->", color=coral, lw=1.2))

ax.set_xlim(0, len(ks)); ax.set_ylim(0, 1.02)
ax.yaxis.set_major_formatter(lambda v, pos: f"{v:.0%}")
ax.xaxis.set_major_formatter(lambda v, pos: f"{int(v):,}")
ax.set_xlabel("Customers contacted, highest risk first")
ax.set_ylabel("Share of churners reached")
ax.legend(frameon=False, loc="lower right", fontsize=10)
ax.grid(axis="x", visible=False)
mult = captured[499] / (500 / len(ks))          # model vs random, at 500 mailers
ax.set_title(f"The top 500 reaches {mult:.1f} times more churners than picking at random",
             loc="left", fontsize=13.5, fontweight="bold", pad=24)
ax.text(0, 1.02, "Backtested on last quarter, n=7,043", transform=ax.transAxes,
        fontsize=10, color=slate)
fig.tight_layout()
fig.savefig("out/backtest_gains.png")

### Do the probabilities mean what they say?

The list we ship cannot be scored against outcomes, so everything rests on the numbers
being honest. Sorting all 7,043 customers into ten equal groups by predicted risk and
comparing prediction against what actually happened is the test.

In [ ]:
dec = pd.qcut(scored["churn_probability"], 10, labels=False, duplicates="drop")
cal = (scored.assign(dec=dec).groupby("dec")
       .agg(n=("Churn Value", "size"), predicted=("churn_probability", "mean"),
            observed=("Churn Value", "mean")))

fig, ax = plt.subplots(figsize=(6.4, 6.0))
ax.plot([0, 0.9], [0, 0.9], color=grey, ls="--", lw=1.5, label="Perfect calibration")
ax.plot(cal["predicted"], cal["observed"], "o-", color=teal, lw=2, ms=8,
        label="Our model")
ax.set_xlim(0, 0.9); ax.set_ylim(0, 0.9)
ax.xaxis.set_major_formatter(lambda v, pos: f"{v:.0%}")
ax.yaxis.set_major_formatter(lambda v, pos: f"{v:.0%}")
ax.set_xlabel("Risk we predicted")
ax.set_ylabel("Share who actually left")
ax.legend(frameon=False, loc="upper left", fontsize=10)
ax.set_title("When we say 70%, 70% of them leave",
             loc="left", fontsize=13.5, fontweight="bold", pad=24)
ax.text(0, 1.02, f"Ten equal groups by predicted risk  |  never more than "
                 f"{(cal.predicted - cal.observed).abs().max():.0%} out",
        transform=ax.transAxes, fontsize=10, color=slate)
fig.tight_layout()
fig.savefig("out/calibration_curve.png")

print(f"largest gap between predicted and observed, across all ten groups: "
      f"{(cal['predicted'] - cal['observed']).abs().max():.1%}")

> **The model can be trusted on two counts.**
>
> **It ranks.** Backtested on last quarter, its top 500 contained **417 people who did
> leave** (83.4%), against 26.5% for contacting people at random. Those 500 mailers would
> have reached **22% of everyone who churned** while touching 7% of the base.
>
> **It is honest about its own confidence.** Across ten groups spanning 0.5% to 78%
> predicted risk, prediction and reality never differ by more than **2 percentage points**.
>
> That second point is what licenses any claim about the list we actually ship, because
> those customers have not churned yet and never can be scored directly.

## 3 · Who can we actually mail?

The brief says the data is a *"Single Customer View of customers over last quarter"*, and
`Churn Value = 1` means that customer left during it. So 1,869 of these rows are people who
have already gone.

You cannot send a retention mailer to somebody who has already left, and the service team
will never be *"talking with"* one of them. Both deliverables are therefore built from the
customers who are still here.

In [ ]:
current = scored[scored["Churn Value"] == 0].copy()
departed = scored[scored["Churn Value"] == 1]

print(f"still with Swan       {len(current):,}")
print(f"left last quarter     {len(departed):,}")
print()
print("If we had ranked all 7,043 and taken the top 500, the list would have been:")
naive = scored.nlargest(500, "churn_probability")
print(f"  {int(naive['Churn Value'].sum())} ex-customers and "
      f"{int((naive['Churn Value']==0).sum())} current ones")
print("  which is a list of people we cannot contact.")

### What this does to the numbers

The historical top 500 scored 83% precision because it was allowed to pick people we
already knew had left. The real list cannot do that, so its quality has to be expressed
differently: as the **calibrated probability** that each person leaves next quarter.

That is exactly what calibration was for. Because a score of 0.7 really does mean a 70%
chance, the average score across the list is a usable estimate of how many will go, and it
can be compared against what a random mailing would achieve.

## 4 · Deliverable 1, the 500 mailers

Top 500 by risk among customers who are still here. The extra columns let the retention
team act on a row without going back to the database.

In [ ]:
COLS = ["rank", "CustomerID", "churn_probability", "risk_band",
        "Contract", "Tenure Months", "Monthly Charges", "Payment Method"]

mailer = current.nlargest(500, "churn_probability").reset_index(drop=True)
mailer["rank"] = mailer.index + 1
mailer = mailer[COLS].copy()
mailer["churn_probability"] = mailer["churn_probability"].round(4)
mailer["Monthly Charges"] = mailer["Monthly Charges"].round(2)

assert len(mailer) == 500
assert mailer["CustomerID"].is_unique
assert mailer["rank"].tolist() == list(range(1, 501))
assert not set(mailer["CustomerID"]) & set(departed["CustomerID"]), "an ex-customer got in"

mailer.to_csv("out/swan_mailer_top500.csv", index=False)
print("wrote out/swan_mailer_top500.csv")
mailer.head()

In [ ]:
top = current.nlargest(500, "churn_probability")
EXPECTED = top["churn_probability"].mean()

print("WHO THEY ARE")
print(f"  risk range        {top['churn_probability'].min():.0%} to {top['churn_probability'].max():.0%}")
print(f"  average risk      {EXPECTED:.1%}")
print(f"  expected to leave {top['churn_probability'].sum():.0f} of the 500 next quarter")
print(f"  the same 500 picked at random would lose "
      f"{current['churn_probability'].mean() * 500:.0f}")
print(f"  so this list is {top['churn_probability'].mean() / current['churn_probability'].mean():.1f}x "
      "better than mailing at random")
print()
print(f"  average tenure          {top['Tenure Months'].mean():.0f} months")
print(f"  average monthly charge  ${top['Monthly Charges'].mean():.2f}")
print(f"  revenue in this list    ${top['Monthly Charges'].sum():,.0f} per month")
print()
print("contract mix (%):")
print((top["Contract"].value_counts(normalize=True) * 100).round(1).to_string())
print("\npayment mix (%):")
print((top["Payment Method"].value_counts(normalize=True) * 100).round(1).to_string())

## 5 · Deliverable 2, churn risk for everyone still here

The brief asks for this so the service team can look up whoever is on the phone. That
includes the 500 being mailed, so all 5,174 current customers are in the file with a flag
saying which ones got a mailer. Filtering on that flag gives the other 4,674 if the client
prefers to read it that way.

In [ ]:
risk = current[["CustomerID", "churn_probability", "risk_band"]].copy()
risk["churn_probability"] = risk["churn_probability"].round(4)
risk["in_mailer_list"] = risk["CustomerID"].isin(mailer["CustomerID"]).astype(int)
risk = risk.sort_values("churn_probability", ascending=False)

assert len(risk) == len(current)
assert risk["in_mailer_list"].sum() == 500
assert not set(risk["CustomerID"]) & set(departed["CustomerID"])

risk.to_csv("out/swan_churn_risk_all.csv", index=False)
print(f"wrote out/swan_churn_risk_all.csv, {len(risk):,} rows "
      f"({risk['in_mailer_list'].sum()} flagged as mailed)")
risk["risk_band"].value_counts().reindex([b[0] for b in BANDS])

In [ ]:
counts = risk["risk_band"].value_counts().reindex([b[0] for b in BANDS])[::-1]
colours = [coral, coral, grey, teal][::-1]

fig, ax = plt.subplots(figsize=(8.5, 3.4))
ax.barh(counts.index, counts.values, color=colours, height=0.62)
for i, v in enumerate(counts.values):
    ax.text(v + counts.max() * 0.015, i, f"{v:,}  ({v/len(risk):.0%})",
            va="center", fontsize=10.5)

ax.set_xlim(0, counts.max() * 1.25)
ax.set_xlabel("Customers")
ax.grid(axis="y", visible=False)
watch = int(counts.get("Critical", 0) + counts.get("High", 0))
ax.set_title(f"{watch:,} of the {len(risk):,} are high risk",
             loc="left", fontsize=13.5, fontweight="bold", pad=24)
ax.text(0, 1.02, f"{len(risk):,} customers still with Swan, by risk band",
        transform=ax.transAxes, fontsize=10, color=slate)
fig.tight_layout()
fig.savefig("out/risk_band_distribution.png")

## 6 · Which sign-up metric to incentivise at $2.50

The client can pay $2.50 for every customer signed up to one metric, and wants to know
which. Six add-ons are candidates.

The comparison has to be fair. Each add-on column has a third level, `No internet service`,
and that is the same 1,526 customers every time. They did not decline the add-on, they were
never able to buy it. Including them measures the internet split rather than the add-on, so
this section uses only customers who could actually buy one.

This section uses all 7,043 rows, because it is measuring an observed relationship rather
than building a list.

In [ ]:
ADDONS = ["Online Security", "Tech Support", "Online Backup",
          "Device Protection", "Streaming TV", "Streaming Movies"]

net = df[df["Internet Service"] != "No"]
print(f"could buy an add-on  : {len(net):,}  (churn {net['Churn Value'].mean():.1%})")
print(f"excluded, no internet: {N - len(net):,}  "
      f"(churn {df[df['Internet Service'] == 'No']['Churn Value'].mean():.1%})")

In [ ]:
rows = []
for a in ADDONS:
    g = net.groupby(a)["Churn Value"].agg(n="size", rate="mean")
    rows.append({"add_on": a,
                 "n_without": int(g.loc["No", "n"]), "churn_without": g.loc["No", "rate"],
                 "n_with": int(g.loc["Yes", "n"]), "churn_with": g.loc["Yes", "rate"],
                 "gap_pp": (g.loc["No", "rate"] - g.loc["Yes", "rate"]) * 100})

gaps = pd.DataFrame(rows).sort_values("gap_pp", ascending=False).reset_index(drop=True)
gaps.style.format({"churn_without": "{:.1%}", "churn_with": "{:.1%}",
                   "gap_pp": "{:.1f}", "n_without": "{:,}", "n_with": "{:,}"})

In [ ]:
t = gaps.sort_values("gap_pp")
y = np.arange(len(t))

fig, ax = plt.subplots(figsize=(9.5, 4.6))
for i, r in enumerate(t.itertuples()):
    ax.plot([r.churn_with, r.churn_without], [i, i], color=rule, lw=3, zorder=1)
ax.scatter(t["churn_with"], y, s=110, color=teal, zorder=2, label="Holds the add-on")
ax.scatter(t["churn_without"], y, s=110, color=coral, zorder=2, label="Does not hold it")

for i, r in enumerate(t.itertuples()):
    ax.text(r.churn_without + 0.014, i, f"{r.gap_pp:.0f}pp gap", va="center", fontsize=10)

ax.set_yticks(y)
ax.set_yticklabels(t["add_on"])
ax.set_xlim(0, 0.58)
ax.xaxis.set_major_formatter(lambda x, pos: f"{x:.0%}")
ax.set_xlabel("Churn rate")
ax.grid(axis="y", visible=False)
ax.legend(frameon=False, loc="lower right", fontsize=10)
ax.set_title("Customers without online security churn at 42%, against 15% with it",
             loc="left", fontsize=13.5, fontweight="bold", pad=24)
ax.text(0, 1.02, f"Internet customers only, n={len(net):,}",
        transform=ax.transAxes, fontsize=10, color=slate)
fig.tight_layout()
fig.savefig("out/addon_gaps.png")

Streaming almost vanishes once the comparison is fair, which is worth knowing before
anyone puts it on a slide. Online security and tech support are the real candidates.

The next question is whether that 27 point gap is really the add-on, or just contract
showing through again. Customers on two-year deals buy more add-ons and churn less anyway.

In [ ]:
print("Online Security gap, held within contract type")
for c in ["Month-to-month", "One year", "Two year"]:
    g = net[net["Contract"] == c].groupby("Online Security")["Churn Value"].agg(["size", "mean"])
    gap = (g.loc["No", "mean"] - g.loc["Yes", "mean"]) * 100
    print(f"  {c:15s} without {g.loc['No','mean']:5.1%} (n={g.loc['No','size']:,})"
          f"   with {g.loc['Yes','mean']:5.1%} (n={g.loc['Yes','size']:,})   {gap:5.1f}pp")

print("\nand held within tenure band")
for b in ["0-6", "7-12", "13-24", "25-48", "49-72"]:
    g = net[net["tenure_band"] == b].groupby("Online Security")["Churn Value"].agg(["size", "mean"])
    gap = (g.loc["No", "mean"] - g.loc["Yes", "mean"]) * 100
    print(f"  {b:>6} months   without {g.loc['No','mean']:5.1%}"
          f"   with {g.loc['Yes','mean']:5.1%}   {gap:5.1f}pp")

In [ ]:
# How many of the mailer list could actually take the offer?
eligible = top[(top["Internet Service"] != "No") & (top["Online Security"] == "No")]
print(f"of the 500 mailers, {len(eligible)} have internet but no online security")
print(f"  so {len(eligible)/500:.0%} of the list can act on this offer")
print(f"  at 20% uptake that is {len(eligible)*0.20:.0f} sign-ups, costing "
      f"${len(eligible)*0.20*2.50:,.0f}")

> **Recommendation: online security.**
>
> Among customers who could buy it, **41.8%** churn without it against **14.6%** with it,
> a **27 point** gap and the widest of the six. The gap survives the control: still **21.5pp**
> inside month-to-month, the group we would actually target, and between 11.6 and 28.5pp in
> every tenure band.
>
> Tech support is a close second at 26.5pp and would be a reasonable alternative. Streaming
> is not a candidate at 3.5pp.
>
> **The caveat belongs on the slide.** This is correlation. Customers who choose a security
> add-on may simply be more engaged, and giving the product to someone who did not want it
> may not have the same effect. Section 8 is how to find out.

## 7 · What the mailer costs and what it has to return

The client gave us three numbers: 500 mailers, 20% uptake, $2.50 per sign-up. Everything
else is an assumption, so it is stated here rather than buried.

We do not know how well the offer works, which is exactly what the analysis cannot tell us.
So the question is inverted: how well would it have to work to be worth doing?

In [ ]:
# Given by the client
N_MAIL, UPTAKE, INCENTIVE = 500, 0.20, 2.50

# Our assumptions
UNIT_COST = 1.00      # print and postage per mailer, not supplied in the brief
HORIZON   = 12        # months of revenue credited to a customer we keep

# From the model. EXPECTED is the average calibrated risk across the list, so it is
# the model's own estimate of what share of these 500 will leave.
ARPU = top["Monthly Charges"].mean()

cost = N_MAIL * UNIT_COST + N_MAIL * UPTAKE * INCENTIVE
value_per_save = ARPU * HORIZON
at_risk_takers = N_MAIL * UPTAKE * EXPECTED

print(f"print and postage         ${N_MAIL * UNIT_COST:,.2f}")
print(f"incentives ({N_MAIL*UPTAKE:.0f} sign-ups)    ${N_MAIL * UPTAKE * INCENTIVE:,.2f}")
print(f"TOTAL CAMPAIGN COST       ${cost:,.2f}")
print()
print(f"average risk on the list  {EXPECTED:.1%}")
print(f"a customer here pays      ${ARPU:.2f} a month")
print(f"keeping one for {HORIZON} months  ${value_per_save:,.2f}")
print()
print(f"of the {N_MAIL*UPTAKE:.0f} who accept, about {at_risk_takers:.0f} were going to leave")
print(f"BREAK EVEN: keep {cost/value_per_save:.2f} of them "
      f"= {cost/value_per_save/at_risk_takers:.1%}")

In [ ]:
rows = []
for unit in [0.75, 1.00, 1.50]:
    for months in [6, 12]:
        for avg_risk in [0.50, EXPECTED, 0.75]:
            c = N_MAIL * unit + N_MAIL * UPTAKE * INCENTIVE
            need = c / (ARPU * months)
            rows.append({"mailer cost": unit, "horizon (months)": months,
                         "avg risk on list": avg_risk, "campaign cost": c,
                         "customers to break even": need,
                         "share of at-risk accepters": need / (N_MAIL * UPTAKE * avg_risk)})
sens = pd.DataFrame(rows)
print(f"Worst case in this grid: keep {sens['customers to break even'].max():.1f} customers "
      f"({sens['share of at-risk accepters'].max():.1%} of at-risk accepters).")
sens.style.format({"mailer cost": "${:.2f}", "avg risk on list": "{:.0%}",
                   "campaign cost": "${:,.0f}", "customers to break even": "{:.2f}",
                   "share of at-risk accepters": "{:.2%}"})

In [ ]:
eff = np.linspace(0, 0.30, 61)
net_return = at_risk_takers * eff * value_per_save - cost
breakeven = cost / (at_risk_takers * value_per_save)

fig, ax = plt.subplots(figsize=(9.5, 4.6))
ax.axhspan(net_return.min(), 0, color=coral, alpha=0.07)
ax.plot(eff, net_return, color=teal, lw=2.5)
ax.axhline(0, color=slate, lw=1)
ax.axvline(breakeven, ls="--", color=coral, lw=1.5)
ax.plot([breakeven], [0], "o", color=coral, ms=8, zorder=5)
ax.annotate(f"breaks even at {breakeven:.1%}\n({cost/value_per_save:.1f} customers kept)",
            xy=(breakeven, 0), xytext=(breakeven + 0.025, net_return.max() * 0.30),
            fontsize=10.5, color=coral,
            arrowprops=dict(arrowstyle="->", color=coral, lw=1.2))

ax.set_xlim(0, 0.30)
ax.xaxis.set_major_formatter(lambda x, pos: f"{x:.0%}")
ax.yaxis.set_major_formatter(lambda y, pos: f"${y/1000:,.0f}k")
ax.set_xlabel("How often the offer keeps an at-risk customer who accepts it")
ax.set_ylabel("Net return")
ax.grid(axis="x", visible=False)
ax.set_title("The campaign pays for itself if it keeps one customer",
             loc="left", fontsize=13.5, fontweight="bold", pad=24)
ax.text(0, 1.02, f"Net return on a ${cost:,.0f} campaign, {HORIZON} month horizon",
        transform=ax.transAxes, fontsize=10, color=slate)
fig.tight_layout()
fig.savefig("out/mailer_breakeven.png")

> **The campaign costs about $750** and one retained customer is worth around $920, so it
> breaks even on fewer than one customer, roughly 1% of the at-risk people who accept.
>
> That holds across every assumption in the grid. The decision is not close, so it does not
> need defending at length. The interesting questions are what to put in the mailer, which
> is section 6, and whether it worked, which is section 8.

## 8 · Telling whether it worked

The economics say send it. They do not say the offer works, because nobody has ever tested
it. A holdout fixes that and costs almost nothing:

1. Take the **top 600** current customers by risk.
2. **Randomly** assign 500 to the mailer and 100 to receive nothing.
3. Compare churn between the two groups next quarter.

Random assignment is the point. Comparing the 20% who accept against the 80% who ignore the
mailer proves nothing, because people who respond to retention offers were already
different.

In [ ]:
from scipy.stats import norm

def smallest_detectable_drop(n_treat, n_ctrl, p_ctrl, power=0.80, alpha=0.05):
    """The smallest true churn reduction this design would reliably pick up."""
    za, zb = norm.ppf(1 - alpha / 2), norm.ppf(power)
    for d in np.arange(0.001, 0.60, 0.001):
        p_treat = p_ctrl - d
        if p_treat <= 0:
            break
        pooled = (p_ctrl * n_ctrl + p_treat * n_treat) / (n_treat + n_ctrl)
        se_null = np.sqrt(pooled * (1 - pooled) * (1 / n_treat + 1 / n_ctrl))
        se_alt = np.sqrt(p_ctrl * (1 - p_ctrl) / n_ctrl + p_treat * (1 - p_treat) / n_treat)
        if za * se_null + zb * se_alt <= d:
            return d
    return np.nan

print(f"the control group should churn at about {EXPECTED:.0%}, the list average\n")
for n_ctrl in [100, 150, 250, 400]:
    d = smallest_detectable_drop(500, n_ctrl, EXPECTED)
    print(f"  control of {n_ctrl:3d}  detects a drop of {d:.1%} or more "
          f"({d/EXPECTED:.0%} relative)")

> **Say the limitation out loud.** With 100 held back, the test only picks up a drop of
> around 15 points. A real but modest effect of 5 points would not show, and reading that as
> failure would be wrong.
>
> | Design | Detects | Cost |
> |---|---|---|
> | 500 mail, 100 control | ~15pp or more | free, 100 fewer mailers |
> | 500 mail, 250 control | ~10pp or more | needs 750 scored |
> | Repeat over 3 campaigns | ~6pp or more | no extra spend, takes 3 quarters |
>
> Recommend the 100 version. It costs nothing and answers the question the retention team
> actually has, which is whether this is worth funding properly.

In [ ]:
print("FILES PRODUCED\n")
for f in sorted(Path("out").glob("*")):
    print(f"  {f}  ({f.stat().st_size/1024:.0f} KB)")

print(f"\nBuilt from {SCORES}")
print(f"  population    {len(current):,} customers still with Swan "
      f"({len(departed):,} excluded, already gone)")
print(f"  mailer list   {len(mailer):,} rows, average risk {EXPECTED:.1%}")
print(f"  risk list     {len(risk):,} rows")
print(f"  incentive     online security, {gaps.iloc[0]['gap_pp']:.0f}pp gap")
print(f"  campaign      ${cost:,.0f}, breaks even on {cost/value_per_save:.1f} customers")